## 0. Pydantic warmup

This exercise will give you a feel of the pydantic library for data validation.

a) Create a BaseModel for a User. It should have a required id (integer) and a required name (string). Instantiate the model with valid data and then with invalid data (e.g., a string for id) to see the ValidationError.

b) Create a BaseModel for a Person with the fields name, age, email, favourite pet. Add appropriate validation in each fields. Tips: you can use built-in EmailStr type in pydantic for validating email. Try out your Person class by instantiating it with different types of values for the fields to see proper validations.

c) Use normal python class to replicate what you have created in b), i.e. create a Person class with proper input validation.

a) Create a BaseModel for a User. It should have a required id (integer) and a required name (string). Instantiate the model with valid data and then with invalid data (e.g., a string for id) to see the ValidationError.

In [ ]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    id: int
    name: str

user1 = User(id=1, name="Pontus")
user1

User(id=1, name='Pontus')

In [4]:
try:
    user2 = User(id="id", name=254)
    print(user2)
except ValidationError as err:
    print(err)


2 validation errors for User
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='id', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing
name
  Input should be a valid string [type=string_type, input_value=254, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


b) Create a BaseModel for a Person with the fields name, age, email, favourite pet. Add appropriate validation in each fields. Tips: you can use built-in EmailStr type in pydantic for validating email. Try out your Person class by instantiating it with different types of values for the fields to see proper validations.

In [7]:
from pydantic import EmailStr, Field
from typing import Literal
import email_validator

# class Pet(str, Enum):
#     dog = "dog"
#     cat = "cat"
#     esko = "esko"

class Person(BaseModel):
    name: str
    age: int = Field(gt=-1, lt=126)
    email: EmailStr
    favorite_pet: Literal["Dog", "Cat", "Esko"]


try:
    person1 = Person(name="Pontus", age=31, email="ponagr", favorite_pet="Esko")
    print(person1)
except ValidationError as err:
    print(err)

1 validation error for Person
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='ponagr', input_type=str]


In [9]:
try:
    person2 = Person(name="Pontus", age=31, email="ponagr@hotmail.com", favorite_pet="Esko")
    print(person2)
except ValidationError as err:
    print(err)

name='Pontus' age=31 email='ponagr@hotmail.com' favorite_pet='Esko'


In [10]:
try:
    person2 = Person(name="Pontus", age=31, email="ponagr@hotmail.com", favorite_pet="Natali")
    print(person2)
except ValidationError as err:
    print(err)

1 validation error for Person
favorite_pet
  Input should be 'Dog', 'Cat' or 'Esko' [type=literal_error, input_value='Natali', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/literal_error


c) Use normal python class to replicate what you have created in b), i.e. create a Person class with proper input validation.

In [29]:
class Person:
    def __init__(self, name: str, age: int, email: str, favorite_pet: str):
        if not isinstance(name, str):
            raise TypeError(f"'name' must be of type str not {type(name)}")
        self.name = name
        self.age = age
        self.email = email
        self.favorite_pet = favorite_pet
        
    # getter
    @property
    def age(self):
        return self._age
    
    # setter, om value klarar alla valideringar, så sätter vi value till age
    @age.setter
    def age(self, value: int):
        if not isinstance(value, int):
            raise TypeError(f"age must be of type int not {type(value)} that you have provided")
        if value < 0 or value > 125:
            raise ValueError(f"age must be between 0 and 125, not {value} that you provided")
        
        self._age = value
    
    @property
    def email(self):
        return self._email

    @email.setter
    def email(self, value: str):
        if not isinstance(value, str):
            raise TypeError(f"email must be of type str not {type(value)} that you have provided")
        if not value.__contains__("@"):
            raise ValueError(f"email must contain '@'")
        if not value.__contains__("."):
            raise ValueError(f"email must contain '.'")
        if value.__contains__(" "):
            raise ValueError(f"email must contain spaces'")
        
        self._email = value
    
    @property
    def favorite_pet(self):
        return self._favorite_pet
    
    @favorite_pet.setter
    def favorite_pet(self, value: str):
        if not isinstance(value, str):
            raise TypeError(f"favorite pet must be of type str not {type(value)} that you have provided")
        if value == "Dog" or value == "Cat" or value == "Esko":
            self._favorite_pet = value
        else:
            raise ValueError("favorite pet must be 'Dog', 'Cat' or 'Esko'")


try:
    person2 = Person(name="Pontus", age=31, email="ponag@hotmail.com", favorite_pet="Esko")
    print(person2)
except ValueError as err:
    print(err)
except TypeError as err1:
    print(err1)